In [ ]:
import requests
from datetime import datetime

def timestamp_to_date(timestamp):
    return datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')


def get_contract_transactions(contract_address, etherscan_api_key):
    # Step 1: Get the list of transactions for the contract address
    etherscan_url = f"https://api.etherscan.io/api?module=account&action=txlist&address={contract_address}&apikey={etherscan_api_key}"
    response = requests.get(etherscan_url)
    if response.status_code == 200:
        data = response.json()
        if 'result' in data and data['result'] is not None:
            transactions = data['result']
            return transactions
        else:
            raise ValueError("Failed to get transactions from Etherscan API.")
    else:
        raise ValueError("Failed to fetch data from Etherscan API.")

def get_contract_creation_timestamp_and_last_tx_timestamp(contract_address, etherscan_api_key):
    transactions = get_contract_transactions(contract_address, etherscan_api_key)

    if transactions:
        # Step 2: Find the contract creation transaction (transaction with an empty 'to' address)
        contract_creation_tx = next((tx for tx in transactions if tx['to'] == ''), None)
        print(contract_creation_tx)
        if contract_creation_tx is not None:
            creation_timestamp = int(contract_creation_tx['timeStamp'])

            # Step 3: Find the last transaction in the list and get its timestamp
            last_tx = transactions[-1]
            last_tx_timestamp = int(last_tx['timeStamp'])

            return creation_timestamp, last_tx_timestamp, len(transactions)
        else:
            raise ValueError("Contract creation transaction not found.")
    else:
        raise ValueError("No transactions found for the contract address.")


In [ ]:
import pandas as pd
def process_contract_addresses(input_file, output_file, etherscan_api_key):
    # Read the total number of rows in the input CSV file
    total_rows = sum(1 for row in open(input_file, 'r'))

    # Calculate the number of rows to read (30% of total)
    rows_to_read = int(total_rows * 0.3)

    # Read the first 30% of rows into a Pandas DataFrame
    df = pd.read_csv(input_file, nrows=rows_to_read)

    # Initialize new columns in the DataFrame to store contract information
    df['Creation Timestamp'] = None
    df['Creation Date'] = None
    df['Last Transaction Timestamp'] = None
    df['Last Transaction Date'] = None
    df['Total Transactions'] = None

    for index, row in df.iterrows():
        contract_address = row['implementations']
        try:
            creation_timestamp, last_tx_timestamp, total_transactions = get_contract_creation_timestamp_and_last_tx_timestamp(contract_address, etherscan_api_key)
            df.at[index, 'Creation Timestamp'] = creation_timestamp
            df.at[index, 'Creation Date'] = timestamp_to_date(creation_timestamp)
            df.at[index, 'Last Transaction Timestamp'] = last_tx_timestamp if last_tx_timestamp else "N/A"
            df.at[index, 'Last Transaction Date'] = timestamp_to_date(last_tx_timestamp) if last_tx_timestamp else "N/A"
            df.at[index, 'Total Transactions'] = total_transactions
        except Exception as e:
            print(f"Error processing contract address {contract_address}: {e}")

    # Save the processed DataFrame to the output CSV file
    df.to_csv(output_file, index=False)

if __name__ == "__main__":
    input_csv_file = "RQ2.csv"  # Replace this with the path to your input CSV file
    output_csv_file = "RQ4.csv"  # Replace this with the desired path for the output CSV file
    etherscan_api_key = ""  # Replace this with your Etherscan API key

    process_contract_addresses(input_csv_file, output_csv_file, etherscan_api_key)
    print("Contract information saved to", output_csv_file)
